# Hybrid retrieval & re-ranking — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cos(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))
print('setup ok')

## Fuse sparse and dense candidates, then re-rank the shortlist
The examples show score normalization, linear fusion, candidate union, re-ranking, and how raw uncalibrated scores can break hybrid retrieval.

In [ ]:
# 1) Normalize sparse and dense scores onto comparable 0-1 scales
bm_raw=np.array([8.,2.,6.,1.]); de_raw=np.array([.1,.9,.5,.3])
norm=lambda x:(x-x.min())/(x.max()-x.min())
bm=norm(bm_raw); de=norm(de_raw)
plt.figure(figsize=(6,3)); plt.bar(np.arange(4)-.15,bm,width=.3,label='BM25 norm'); plt.bar(np.arange(4)+.15,de,width=.3,label='dense norm'); plt.xticks(range(4),['d1','d2','d3','d4']); plt.legend(); plt.title('normalize before fusion')
assert bm.max()==1 and de.max()==1

In [ ]:
# 2) Linear hybrid fusion balances exact and semantic evidence
bm=np.array([0.80,0.20,0.60,0.10]); de=np.array([0.10,0.90,0.50,0.30]); alpha=.5; hy=alpha*bm+(1-alpha)*de
plt.figure(figsize=(6,3)); plt.bar(['d1','d2','d3','d4'],hy); plt.ylabel('hybrid score'); plt.title('d2 and d3 tie after fusion')
assert np.allclose(hy,[0.45,0.55,0.55,0.2])

In [ ]:
# 3) Candidate union preserves sparse-only and dense-only finds
sparse_top={0,2}; dense_top={1,2}; union=sorted(sparse_top|dense_top)
plt.figure(figsize=(6,3)); plt.bar(['sparse top','dense top','union'],[len(sparse_top),len(dense_top),len(union)]); plt.title('union raises candidate recall')
assert union==[0,1,2]

In [ ]:
# 4) Re-ranking the shortlist can revise the final order
rerank=np.array([0.30,0.95,0.55,0.20]); final=.4*hy+.6*rerank
plt.figure(figsize=(6,3)); plt.bar(['d1','d2','d3','d4'],final); plt.ylabel('final score'); plt.title('re-ranker lifts d2 above the tie')
assert np.allclose(final,[0.36,0.79,0.55,0.2]) and int(np.argmax(final))==1

In [ ]:
# 5) Raw score averaging lets BM25 scale dominate
raw_avg=(bm_raw+de_raw)/2
plt.figure(figsize=(6,3)); plt.bar(['d1','d2','d3','d4'],raw_avg); plt.ylabel('raw average'); plt.title('uncalibrated fusion overweights sparse scale')
assert raw_avg[0]>raw_avg[1] and hy[1]>hy[0]